# 4.1 – Question 5: Additional insights (options for the third insight)

> **Nota:** abaixo estão 3 insights alternativos (A, B, C). Escolhe **um** e apaga os outros. Todos usam o `artist_df` já limpo e os imports do notebook (`pandas`, `numpy`, `plotly.express as px`, `plotly.graph_objects as go`).
> O Insight C precisa de mais um import: `from plotly.subplots import make_subplots`.

---
## Insight A – Is collaborating worth it? It depends on the genre

**Question:** The overall relationship between collaboration and success can be misleading, because genres have very different collaboration cultures (e.g. Hip-Hop and Reggaeton collaborate a lot, Rock very little). Does a higher share of collaborative streams actually go together with more total streams **once we compare artists within the same genre**?

**Approach and justification:**
- `Collab %` = Collaborative Streams / Total Streams × 100.
- To control for the genre effect, artists are split into three collaboration levels (*Low / Medium / High*) using **tertiles computed within each genre**. Ranking before `qcut` avoids errors in genres where many artists share the same value.
- Only genres with **at least 25 artists** are kept, so that each group has enough observations for a median to be meaningful. We use the **median** because stream counts are right-skewed.
- A Spearman correlation per genre is added as a second, scale-free check.

In [ ]:
# Percentagem de streams que vem de colaborações
artist_df['Collab %'] = artist_df['Collaborative Streams'] / artist_df['Total Streams'] * 100

# Só géneros com pelo menos 25 artistas, para que cada grupo tenha dimensão razoável
genre_counts = artist_df['Primary Genre'].value_counts()
main_genres = genre_counts[genre_counts >= 25].index.tolist()
genre_df = artist_df[artist_df['Primary Genre'].isin(main_genres)].copy()
genre_df['Primary Genre'] = genre_df['Primary Genre'].astype(str)

# Dividir cada género em 3 níveis de colaboração (tercis calculados DENTRO do género)
# rank(method='first') evita erros quando há muitos artistas com a mesma % (ex.: Rock com ~0%)
labels = ['Low', 'Medium', 'High']
genre_df['Collab Level'] = genre_df.groupby('Primary Genre')['Collab %'].transform(
    lambda s: pd.qcut(s.rank(method='first'), 3, labels=labels)
)
genre_df['Collab Level'] = pd.Categorical(genre_df['Collab Level'], categories=labels, ordered=True)

print(f"Géneros analisados ({len(main_genres)}): {main_genres}")
print(f"Artistas incluídos: {len(genre_df)} de {len(artist_df)}")

In [ ]:
# Mediana de streams por género e nível de colaboração
heat = (genre_df.pivot_table(index='Primary Genre', columns='Collab Level', values='Total Streams',
                             aggfunc='median', observed=True)[labels].round(0))
heat = heat.loc[heat.mean(axis=1).sort_values(ascending=False).index]

# Intervalo de % colaboração que define cada nível (para legendar o gráfico)
bounds = genre_df.groupby(['Primary Genre', 'Collab Level'], observed=True)['Collab %'].agg(['min', 'max']).round(0)
display(heat)

fig = px.imshow(heat, text_auto=',.0f', aspect='auto', color_continuous_scale='Greens',
                labels=dict(x='Collaboration level (within genre)', y='', color='Median streams (M)'),
                title='Median Total Streams by Genre and Collaboration Level')
fig.update_layout(height=450)
fig.show()

**Reading the heatmap:** each row compares artists of the *same* genre, so differences across columns are not caused by genre. Colour intensity shows the median Total Streams (in millions).

In [ ]:
# Ganho relativo de ir de "Low" para "High" colaboração, dentro de cada género
gain = ((heat['High'] / heat['Low'] - 1) * 100).round(1).sort_values()

# Correlação de Spearman (Collab % vs Total Streams) em cada género
spearman = genre_df.groupby('Primary Genre').apply(
    lambda g: g['Collab %'].corr(g['Total Streams'], method='spearman')).round(2)

summary = pd.DataFrame({'High vs Low (% change in median streams)': gain,
                        'Spearman (Collab % vs Total Streams)': spearman}).sort_values(
                        'High vs Low (% change in median streams)', ascending=False)
display(summary)

fig = px.bar(summary.reset_index(), x='High vs Low (% change in median streams)', y='Primary Genre',
             orientation='h', text='High vs Low (% change in median streams)',
             color_discrete_sequence=['#4EA72E'],
             title='Does collaborating more pay off? Change in median streams, High vs Low collaboration')
fig.update_traces(texttemplate='%{text:+.0f}%')
fig.update_layout(yaxis_title='', xaxis_title='% change in median streams (High vs Low)', height=420)
fig.add_vline(x=0, line_color='black')
fig.show()

**Conclusion – Insight A:**
- Collaboration is **not a universal growth strategy**. In Pop and Latin the most collaborative artists have a slightly higher median (+8% and +6%), but the Spearman correlations are close to zero, so the effect is weak.
- **Reggaeton is the clear exception**: artists in the highest collaboration tertile have a median ~49% lower than those in the lowest (Spearman ≈ −0.53). In this genre, success is associated with a *solo* catalogue, not with features.
- In Hip-Hop, R&B and Rock the difference is small (between −9% and 0%), so collaboration neither helps nor hurts noticeably.
- Medium collaboration often has the best median (Latin, R&B, Pop), which suggests a balance rather than extremes.

**Limitations:** this is correlation, not causation (big stars may be asked for fewer features, or may not need them), and the sample is only the Top 500, with 25–115 artists per genre.

---
## Insight B – How unequal is success? Concentration of streams (Lorenz curve and Gini)

**Question:** Do a few superstars capture most of the streams, or is success spread out even among the Top 500? And is this inequality the same in every genre?

**Approach and justification:**
- The **Lorenz curve** plots the cumulative share of artists (from smallest to largest) against the cumulative share of streams. The further it bends away from the diagonal, the more concentrated the streams.
- The **Gini index** summarises this in one number (0 = perfect equality, 1 = one artist has everything). It is implemented with numpy as a small reusable function, so it can be applied by genre with `groupby`.
- Genre comparison uses genres with **at least 20 artists**, so the index is not driven by one or two artists.

In [ ]:
def gini(values):
    """Índice de Gini: 0 = streams igualmente distribuídos; 1 = um artista tem tudo."""
    v = np.sort(np.asarray(values, dtype='float64'))
    n = len(v)
    cum = np.cumsum(v)
    return (n + 1 - 2 * (cum / cum[-1]).sum()) / n

# Curva de Lorenz: % acumulada de artistas (do mais pequeno ao maior) vs % acumulada de streams
sorted_streams = np.sort(artist_df['Total Streams'].values.astype('float64'))
lorenz_y = np.insert(np.cumsum(sorted_streams) / sorted_streams.sum(), 0, 0)
lorenz_x = np.linspace(0, 1, len(lorenz_y))

total_gini = gini(artist_df['Total Streams'])

# Quantos artistas concentram 50% dos streams?
desc = np.sort(artist_df['Total Streams'].values.astype('float64'))[::-1]
cum_share = np.cumsum(desc) / desc.sum()
n50 = int((cum_share < 0.5).sum() + 1)
top10_share = desc[:10].sum() / desc.sum() * 100
top50_share = desc[:50].sum() / desc.sum() * 100   # top 10% dos artistas

print(f"Gini dos streams totais: {total_gini:.3f}")
print(f"Top 10 artistas concentram {top10_share:.1f}% dos streams")
print(f"Top 50 artistas (10%) concentram {top50_share:.1f}% dos streams")
print(f"São precisos {n50} artistas ({n50/len(desc)*100:.1f}%) para atingir 50% dos streams")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Perfect equality',
                         line=dict(color='grey', dash='dash')))
fig.add_trace(go.Scatter(x=lorenz_x, y=lorenz_y, mode='lines', name=f'Lorenz curve (Gini = {total_gini:.2f})',
                         line=dict(color='#2E6B1A', width=3), fill='tonexty', fillcolor='rgba(78,167,46,0.25)'))
fig.update_layout(title='Concentration of Streams among the Top 500 Artists (Lorenz Curve)',
                  xaxis_title='Cumulative share of artists (smallest to largest)',
                  yaxis_title='Cumulative share of total streams',
                  xaxis_tickformat='.0%', yaxis_tickformat='.0%', height=520, width=700)
fig.show()

In [ ]:
# Gini por género (só géneros com pelo menos 20 artistas)
genres20 = artist_df['Primary Genre'].value_counts()
genres20 = genres20[genres20 >= 20].index
gini_genre = (artist_df[artist_df['Primary Genre'].isin(genres20)]
              .groupby('Primary Genre', observed=True)['Total Streams']
              .agg(Gini=gini, Artists='count', Top1_share=lambda s: s.max() / s.sum() * 100)
              .round(2).sort_values('Gini', ascending=False))
display(gini_genre)

fig = px.bar(gini_genre.reset_index(), x='Gini', y='Primary Genre', orientation='h', text='Gini',
             color='Gini', color_continuous_scale='Greens', hover_data=['Artists', 'Top1_share'],
             title='Inequality of Streams within each Genre (Gini index)')
fig.update_layout(yaxis=dict(autorange='reversed'), yaxis_title='', coloraxis_showscale=False, height=450)
fig.add_vline(x=total_gini, line_dash='dash', line_color='black',
              annotation_text=f'All artists: {total_gini:.2f}')
fig.show()

**Conclusion – Insight B:**
- Even among the world's top 500 artists, streams are concentrated: the **Gini is ≈ 0.36**, the **top 10% of artists (50) hold ~31% of the streams**, and only **~24% of artists are needed to reach half of all streams**.
- However, the **top 10 artists hold only ~11%**, so there is no single dominant superstar: the elite is relatively broad.
- Inequality differs by genre: **Reggaeton (0.41) and Pop (0.40)** are the most unequal, while **Rock, Latin and EDM (≈ 0.26–0.27)** are far more balanced. In R&B, a single artist accounts for ~17% of the genre's streams.

**Limitations:** the dataset only contains the Top 500, so this measures inequality **among already successful artists**. The inequality in the whole music industry would be much higher.

---
## Insight C – Does when you debut matter? Debut-decade cohorts

**Question:** Do artists who debuted in a given decade accumulate more streams? And how has the genre mix of successful artists changed over time?

**Approach and justification:**
- `Debut Year` is grouped into decades with `pd.cut` (everything before 1990 is merged because there are few artists). The 2020s cohort is very small (14 artists) and must be read with caution.
- We compare **Total Streams** and **Streams per Year of career**. Boxplots use a **log scale** because streams are heavily right-skewed.
- A decade × genre table of counts (top 8 genres) shows how the genre mix shifted.
- This connects with the conclusion of Question 4a: *when* a career happens matters more than how long it is.

In [ ]:
from plotly.subplots import make_subplots

In [ ]:
# Década de estreia (agrupamos tudo o que é anterior a 1990 numa só categoria, pois há poucos artistas)
decade_labels = ['Before 1990', '1990s', '2000s', '2010s', '2020s']
artist_df['Debut Decade'] = pd.cut(artist_df['Debut Year'], bins=[0, 1989, 1999, 2009, 2019, 2100],
                                   labels=decade_labels)

decade_stats = artist_df.groupby('Debut Decade', observed=True).agg(
    Artists=('Total Streams', 'count'),
    Median_Total=('Total Streams', 'median'),
    Median_per_Year=('Streams per Year', 'median'),
    Mean_Total=('Total Streams', 'mean')).round(0)
decade_stats['Share of artists (%)'] = (decade_stats['Artists'] / len(artist_df) * 100).round(1)
display(decade_stats)

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Total Streams by Debut Decade',
                                                    'Streams per Year of Career by Debut Decade'))
for col_i, col in enumerate(['Total Streams', 'Streams per Year'], start=1):
    for dec in decade_labels:
        sub = artist_df[artist_df['Debut Decade'] == dec]
        fig.add_trace(go.Box(y=sub[col], name=dec, marker_color='#4EA72E', showlegend=False,
                             boxpoints=False, hovertext=sub['Artist Name']), row=1, col=col_i)
fig.update_yaxes(type='log', title_text='Streams (millions, log scale)', row=1, col=1)
fig.update_yaxes(type='log', title_text='Streams per year (millions, log scale)', row=1, col=2)
fig.update_layout(height=480)
fig.show()

In [ ]:
# Como mudou o "gosto"? Nº de artistas por género e década de estreia (top 8 géneros)
top_genres = artist_df['Primary Genre'].value_counts().head(8).index
cohort = (artist_df[artist_df['Primary Genre'].isin(top_genres)]
          .groupby(['Primary Genre', 'Debut Decade'], observed=True).size()
          .unstack(fill_value=0).reindex(columns=decade_labels, fill_value=0))
cohort = cohort.loc[top_genres]

fig = px.imshow(cohort, text_auto=True, aspect='auto', color_continuous_scale='Greens',
                labels=dict(x='Debut decade', y='', color='Number of artists'),
                title='Which genres debuted in which decade? (top 8 genres)')
fig.update_layout(height=450)
fig.show()

**Conclusion – Insight C:**
- **Total streams are similar across cohorts** (median ≈ 9.6–12.8 thousand million in every decade): debuting earlier does not by itself lead to more total streams. Artists who debuted in the 2000s and 2010s have the highest medians.
- **Streams per year of career grow steadily by cohort** (≈ 195 → 401 → 614 → 1,094 → 1,981 million per year from "Before 1990" to the 2020s). Newer artists accumulate streams much faster, which reflects the streaming era (but also partly arithmetic: a short career divides the same total by fewer years).
- The genre mix changed a lot: **Rock was dominated by pre-1990 debuts** (28 of 60), while **Hip-Hop, Pop, Reggaeton and R&B are mostly 2000s–2010s** debuts. Regional Mexican and EDM are almost entirely recent.

**Limitations:** survivorship bias (only artists who are still big enough today are in the Top 500; many successful older artists are missing), and the 2020s cohort is tiny.